# 02 — Volatility Analysis
Computes log returns, rolling volatility, and correlation across the three currency pairs.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.data_pipeline import load_all
from ml.data_pipeline.preprocessing import clean, add_features

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 4)

raw = load_all()
processed = {name: add_features(clean(df)) for name, df in raw.items()}
print('Pairs loaded:', list(processed.keys()))

## 1. Rolling Volatility (30-day & 60-day)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, (name, df) in zip(axes, processed.items()):
    ax.plot(df['Date'], df['rolling_vol_30'], label='30-day vol', linewidth=1)
    ax.plot(df['Date'], df['rolling_vol_60'], label='60-day vol', linewidth=1, alpha=0.7)
    ax.set_title(f'{name.replace("_", "/")} — Rolling Volatility')
    ax.set_ylabel('Std of Log Returns')
    ax.legend()

plt.tight_layout()
plt.savefig('../data/processed/volatility.png', dpi=150)
plt.show()

## 2. Log Return Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, df) in zip(axes, processed.items()):
    ax.hist(df['log_return'], bins=80, edgecolor='none', alpha=0.8)
    mean = df['log_return'].mean()
    std  = df['log_return'].std()
    ax.axvline(mean, color='red',    linestyle='--', label=f'mean={mean:.5f}')
    ax.axvline(mean + 2*std, color='orange', linestyle=':', label=f'+2σ')
    ax.axvline(mean - 2*std, color='orange', linestyle=':')
    ax.set_title(f'{name.replace("_", "/")} Log Returns')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Volatility Statistics Summary

In [ ]:
rows = []
for name, df in processed.items():
    rows.append({
        'Pair':         name.replace('_', '/'),
        'Mean Return':  df['log_return'].mean(),
        'Std Return':   df['log_return'].std(),
        'Annualised Vol': df['log_return'].std() * np.sqrt(252),
        'Max Vol (30d)': df['rolling_vol_30'].max(),
        'Min Vol (30d)': df['rolling_vol_30'].min(),
    })

summary = pd.DataFrame(rows).set_index('Pair')
summary.style.format('{:.6f}')

## 4. Correlation Matrix

In [ ]:
log_returns = pd.DataFrame({
    name.replace('_', '/'): df.set_index('Date')['log_return']
    for name, df in processed.items()
}).dropna()

corr = log_returns.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Log Return Correlation Matrix')
plt.tight_layout()
plt.savefig('../data/processed/correlation_matrix.png', dpi=150)
plt.show()
print(corr.to_string())

## 5. High-Volatility Periods

In [ ]:
# Flag dates where 30-day vol exceeds the 90th percentile for each pair
for name, df in processed.items():
    threshold = df['rolling_vol_30'].quantile(0.90)
    high_vol  = df[df['rolling_vol_30'] > threshold]
    print(f"{name}: {len(high_vol)} high-vol days  (threshold={threshold:.6f})")
    print(f"  Earliest: {high_vol['Date'].min().date()}  Latest: {high_vol['Date'].max().date()}")